# E13 — training: the context arm, then N2a's two diagnostic arms

Registration: `research/e13/PREREG_E13.md` §§2.1, 2.5.

| arm | what |
|---|---|
| `e13_ctx` | `e12_final`'s recipe + **ContextSwap**: for the phishing query rows, half the draws put a host and path `e12_final` wrote before the real query; the model-written part is input only, never a target |
| N `A1` | `e12_final`'s recipe on query rows only, each row at `e12_final`'s exposure (diagnostic) |
| N `A2` | the same at four times the exposure (diagnostic) |

**Before any training, cell 3 rebuilds E11's start and checks it**, and checks that a
384-token step fits. If either fails it stops before a session is spent.

**Attach two inputs** (right panel → *Add Input*):
1. **`uie-corpus2`**, the data bundle E7–E12 used;
2. **`uie-e13-code`**.

**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About 2.5 h.
**A rerun skips a finished arm.** When done, download `/kaggle/working`.

In [ ]:
# --- 1. locate the code bundle -----------------------------------------------------
import hashlib, json
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
code_hits = sorted(root.rglob("E13_CODE_MANIFEST.json")) if root.exists() else []
if not code_hits:
    raise SystemExit("Attach the uie-e13-code dataset (right panel -> Add Input), then rerun.")
CODE = code_hits[0].parent
WORK = Path("/kaggle/working")
print("code:", CODE)

data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
if not data_hits:
    raise SystemExit("Attach the uie-corpus2 dataset too, then rerun.")
DATA = data_hits[0].parent
print("data:", DATA)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
for p in (CODE / "research" / "e13", CODE / "research" / "native", CODE):
    sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")
             or m in ("e8_common", "e11_common", "e12_common", "e13_common", "e13n_train")]:
    del sys.modules[name]
import uie
import e12_common as K
import e13_common as E13
manifest = json.loads((CODE / "E13_CODE_MANIFEST.json").read_text())
for mod in (uie, K, E13):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
if sha256(CODE / "e12_final.pt") != manifest["e12_final"]:
    bad.append("e12_final.pt")
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files, "
      f"{len(manifest['tables'])} tables and e12_final.pt verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
print("torch:", torch.__version__, "| gpu:", torch.cuda.get_device_name(0))

need = ["name_train", "structure_train", "phishing_name_train", "phishing_structure_train",
        "name_val", "structure_val", "phishing_name_val", "phishing_structure_val"]
present = {p.name for p in DATA.iterdir()}
missing = [s for s in need if f"{s}.jsonl" not in present and f"{s}.jsonl.gz" not in present]
if missing:
    raise SystemExit(f"STOP. missing {missing}: this is not the uie-corpus2 bundle.")
import inspect
from uie.kaggle_train import run
assert {"context_prefixes", "context_p", "slash_plan", "max_len", "e12_start"} <= set(
    inspect.signature(run).parameters)
import e13n_train as N
assert N.EXPECTED_KEPT and N.E12_FINAL_SHA256 == manifest["e12_final"], "N's counts not registered"
print("data bundle: all 8 streams present; E13 options present")
print("e13_ctx:", E13.recipe_ctx(CODE / E13.PREFIX_FILE))

In [ ]:
# --- 3. THE START CHECK and the memory preflight, before any training -----------------
e10, e11 = K.start_fingerprints()
print("E10 start:", e10, "(recorded", K.E10_START + ")")
print("E11 start:", e11, "(recorded", K.E11_START + ")")
if (e10, e11) != (K.E10_START, K.E11_START):
    raise SystemExit("STOP. This torch rebuilds a different start than E10/E11 recorded. "
                     "Nothing was trained.")
try:
    peak = K.memory_preflight(torch.device("cuda"))
except torch.cuda.OutOfMemoryError:
    raise SystemExit("STOP. A full 256 x 384 training step does not fit on this GPU.")
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"start check PASSED; memory preflight PASSED ({peak:.1f} of {total:.1f} GB)")

In [ ]:
# --- 4. train e13_ctx (skipped if finished) -----------------------------------------
import time

PROV = WORK / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"arms": {}}
prov.update({"registration": E13.REGISTRATION, "code_commit": manifest["commit"],
             "start_check": {"e10": e10, "e11": e11},
             "prefixes_sha256": manifest["tables"][E13.PREFIX_FILE]})
out = WORK / "e13_ctx"
if (out / "train_result.json").exists() and (out / "final.pt").exists():
    print("e13_ctx: finished earlier -- skipped")
else:
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    run(DATA, out, **E13.recipe_ctx(CODE / E13.PREFIX_FILE))
    prov["arms"]["e13_ctx"] = {"wall_minutes": round((time.time() - t0) / 60, 1)}
    PROV.write_text(json.dumps(prov, indent=1))

In [ ]:
# --- 5. read e13_ctx back: settings, the start (V-id) and ContextSwap (V5) ------------
problems = []
ck = torch.load(WORK / "e13_ctx" / "final.pt", map_location="cpu", weights_only=False)
tc, mc, tk = ck["train_config"], ck["model_config"], ck["tokenizer"]
r = json.loads((WORK / "e13_ctx" / "train_result.json").read_text())
rec = E13.recipe_ctx(CODE / E13.PREFIX_FILE)
s = r.get("e12_start") or {}
cs = r.get("context_swap") or {}
want = [("steps", tc.get("steps"), 20000), ("batch_size", tc.get("batch_size"), 256),
        ("seed", tc.get("seed"), 20260803), ("loss_norm", tc.get("loss_norm"), "fixed"),
        ("loss_constant", round(r.get("loss_constant") or 0, 3), round(rec["loss_constant"], 3)),
        ("grad_clip_late", tc.get("grad_clip_late"), 5.0), ("max_len", mc.get("max_len"), 384),
        ("vocab_size", mc.get("vocab_size"), 110), ("slash_plan", tk.get("slash_plan"), True),
        ("context_p", tc.get("context_p"), E13.CONTEXT_P),
        ("context_seed", tc.get("context_seed"), E13.CONTEXT_SEED),
        ("init weights (V-id)", r.get("init_weights_sha256"), E13.E12_FINAL_INIT)]
want += [(f"start: {k}", s.get(k), v) for k, v in E13.E12_FINAL_START.items()]
want += [("matched rows (V5)", cs.get("matched_rows"), cs.get("listed_rows"))]
wrong = [w for w in want if w[1] != w[2]]
problems += wrong
drawn = cs.get("eligible_draws") or 0
decided = (cs.get("swapped") or 0) + (cs.get("too_long") or 0)
share = decided / drawn if drawn else float("nan")
band = 3 * (0.25 / drawn) ** 0.5 if drawn else float("nan")
if not abs(share - E13.CONTEXT_P) <= band:
    problems.append(("swap share (V5)", share, f"{E13.CONTEXT_P} +- {band:.4f}"))
prov["arms"].setdefault("e13_ctx", {})
prov["arms"]["e13_ctx"].update({"best_sha256": sha256(WORK / "e13_ctx" / "best.pt"),
                                "final_sha256": sha256(WORK / "e13_ctx" / "final.pt"),
                                "init_weights_sha256": r.get("init_weights_sha256"),
                                "context_swap": cs, "final_val": r.get("final"),
                                "counts": r.get("counts")})
PROV.write_text(json.dumps(prov, indent=1))
print(f"e13_ctx val {r.get('final')}; ContextSwap {cs}; swap share {share:.4f} (band +- {band:.4f})")
print("MISMATCHES:", problems if problems else "none -- e13_ctx is as registered")

In [ ]:
# --- 6. N: A1 then A2, each trained from E12's start, then its query sets ------------
for arm in ("A1", "A2"):
    out = WORK / f"n_{arm}"
    if (out / "N_RECORD.json").exists():
        print(f"N {arm}: finished earlier -- skipped")
        continue
    t0 = time.time()
    rec = N.run(DATA, CODE / "e12_final.pt", arm, out, CODE)
    prov["arms"][f"n_{arm}"] = {"final_sha256": rec["final_sha256"], "schedule": rec["schedule"],
                                "kept_query_rows": rec["kept_query_rows"],
                                "wall_minutes": round((time.time() - t0) / 60, 1)}
    PROV.write_text(json.dumps(prov, indent=1))
print("\nall arms trained. Now download /kaggle/working (e13_ctx, n_A1, n_A2, PROVENANCE.json).")